# Logistic Regression — Base vs Google Trends

**Part 1** trains and evaluates a logistic regression on price + engineered features.  
**Part 2** adds 5 Google Trends features and evaluates independently.  
**Part 3** compares both models head-to-head.

In [1]:
import pathlib
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss,
    f1_score, log_loss, precision_recall_curve, roc_auc_score,
)
from sklearn.model_selection import GridSearchCV
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
ROOT            = pathlib.Path("../..")
DATA_DIR        = ROOT / "data"
ARTIFACTS_DIR   = pathlib.Path("artifacts")
PREDICTIONS_DIR = pathlib.Path("predictions")

NUMERIC_FEATURES = [
    "price_at_snapshot",
    "price_deviation_from_half",
    "days_before_close",
    "pct_lifetime_elapsed",
    "duration_days",
    "log_volume",
    "price_mean_7d",   "price_volatility_7d",  "price_min_7d",  "price_max_7d",
    "price_change_7d", "price_range_7d",        "price_trend_7d",
    "price_mean_14d",  "price_volatility_14d", "price_min_14d", "price_max_14d",
    "price_change_14d","price_range_14d",       "price_trend_14d",
]
TRENDS_FEATURES      = ["trend_value", "trend_ma4", "trend_change_4w", "trend_spike", "has_trend_data"]
CATEGORICAL_FEATURES = ["category"]
TARGET               = "outcome"

FEATURES_BASE   = NUMERIC_FEATURES + CATEGORICAL_FEATURES
FEATURES_TRENDS = NUMERIC_FEATURES + TRENDS_FEATURES + CATEGORICAL_FEATURES

---
## Load Data

The trends-enriched dataset contains all base features plus the 5 trend columns, split across two parquet files.

In [3]:
df = pd.read_parquet(DATA_DIR / "polymarket_ml_dataset_with_trends_clean.parquet")

df["category"] = df["category"].fillna("other")
df = df.dropna(subset=[TARGET])

train = df[df["split"] == "train"]
test  = df[df["split"] == "test"]

assert len(set(train["market_id"]) & set(test["market_id"])) == 0, "Market leakage detected"

# 20% of train markets held out for isotonic calibration
rng            = np.random.default_rng(42)
all_market_ids = train["market_id"].unique()
cal_ids        = set(rng.choice(all_market_ids, size=int(len(all_market_ids) * 0.20), replace=False))

train_fit = train[~train["market_id"].isin(cal_ids)]
train_cal = train[ train["market_id"].isin(cal_ids)]

y_train = train_fit[TARGET]
y_cal   = train_cal[TARGET].values
y_test  = test[TARGET]

counts           = train_fit.groupby("market_id").size()
snapshot_weights = train_fit["market_id"].map(counts).rdiv(1).values
class_weights    = compute_sample_weight("balanced", y_train)
sample_weights   = class_weights * snapshot_weights
sample_weights   = sample_weights / sample_weights.mean()
test_reset = test.reset_index(drop=True)

print(f"Total rows : {len(df):,}  |  Columns: {df.shape[1]}")
print(f"Train fit  : {len(train_fit):,}  |  {train_fit['market_id'].nunique():,} markets")
print(f"Train cal  : {len(train_cal):,}   |  {train_cal['market_id'].nunique():,} markets (isotonic calibration)")
print(f"Test       : {len(test):,}   |  {test['market_id'].nunique():,} markets")
print(f"Trend coverage (has_trend_data=1): {df['has_trend_data'].mean():.1%}")
df.head()

Total rows : 1,448,142  |  Columns: 32
Train fit  : 929,967  |  13,420 markets
Train cal  : 229,685   |  3,354 markets (isotonic calibration)
Test       : 288,490   |  4,174 markets
Trend coverage (has_trend_data=1): 99.6%


,market_id,snapshot_timestamp,days_before_close,pct_lifetime_elapsed,duration_days,price_at_snapshot,price_deviation_from_half,total_volume,log_volume,outcome,...,price_range_14d,price_trend_14d,split,category,question,trend_value,trend_ma4,trend_change_4w,trend_spike,has_trend_data
0,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-01 18:45:42.437000+00:00,59.22,0.1912,73,0.03,0.47,40175.18,10.601,0,...,0.05,0.000526,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1
1,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-02 06:45:42.437000+00:00,58.72,0.1980,73,0.03,0.47,40175.18,10.601,0,...,0.05,0.000354,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1
2,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-02 18:45:42.437000+00:00,58.22,0.2049,73,0.03,0.47,40175.18,10.601,0,...,0.05,0.000215,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1
3,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-03 06:45:42.437000+00:00,57.72,0.2117,73,0.03,0.47,40175.18,10.601,0,...,0.03,0.000479,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1
4,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-03 18:45:42.437000+00:00,57.22,0.2185,73,0.03,0.47,40175.18,10.601,0,...,0.03,0.000499,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1


---
## Shared Helpers

In [4]:
def build_pipeline(num_cols, cat_cols, C=1.0, solver="lbfgs", penalty="l2"):
    return Pipeline([
        ("preprocessor", ColumnTransformer([
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
                ("scaler",  StandardScaler()),
            ]), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ])),
        ("clf", LogisticRegression(
            max_iter=1000,
            solver=solver, C=C, penalty=penalty, random_state=42,
        )),
    ])

def get_threshold(y_true, y_prob):
    prec, rec, thresh = precision_recall_curve(y_true, y_prob)
    f1 = 2 * prec * rec / (prec + rec + 1e-9)
    return float(thresh[np.argmax(f1)]), float(np.max(f1))

def evaluate(y_true, y_prob, label, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    metrics = {
        "AUC-ROC" : roc_auc_score(y_true, y_prob),
        "PR-AUC"  : average_precision_score(y_true, y_prob),
        "Log-loss": log_loss(y_true, y_prob),
        "Brier"   : brier_score_loss(y_true, y_prob),
        "Accuracy": accuracy_score(y_true, y_pred),
        "F1"      : f1_score(y_true, y_pred),
    }
    print(f"\n{'─'*50}")
    print(f"  {label}  (threshold={threshold:.3f})")
    print(f"{'─'*50}")
    for name, val in metrics.items():
        print(f"  {name:<10}: {val:.4f}")
    print(f"{'─'*50}")
    return metrics

def per_category(test_df, y_prob, threshold):
    rows = []
    for cat in sorted(test_df["category"].unique()):
        mask = test_df["category"] == cat
        yt   = test_df.loc[mask, TARGET].values
        if len(np.unique(yt)) < 2 or len(yt) < 10:
            continue
        yp = y_prob[mask.values]
        rows.append({
            "category": cat,
            "n"       : int(mask.sum()),
            "AUC"     : roc_auc_score(yt, yp),
            "PR-AUC"  : average_precision_score(yt, yp),
            "F1"      : f1_score(yt, (yp >= threshold).astype(int)),
            "YES%"    : float(yt.mean()),
        })
    return pd.DataFrame(rows).set_index("category").sort_values("AUC", ascending=False)

def market_eval(test_df, y_prob, threshold):
    mdf = (
        test_df.assign(pred_prob=y_prob)
        .groupby("market_id")
        .agg(pred_prob=("pred_prob", "mean"), outcome=(TARGET, "first"))
        .reset_index()
    )
    mp, mt = mdf["pred_prob"].values, mdf["outcome"].values
    mpred  = (mp >= threshold).astype(int)
    print(f"Market-level evaluation ({len(mdf):,} markets)")
    print(f"  AUC-ROC  : {roc_auc_score(mt, mp):.4f}")
    print(f"  PR-AUC   : {average_precision_score(mt, mp):.4f}")
    print(f"  Brier    : {brier_score_loss(mt, mp):.4f}")
    print(f"  Accuracy : {accuracy_score(mt, mpred):.4f}")
    print(f"  F1       : {f1_score(mt, mpred):.4f}")
    return {"AUC-ROC": roc_auc_score(mt,mp), "PR-AUC": average_precision_score(mt,mp),
            "Brier": brier_score_loss(mt,mp), "Accuracy": accuracy_score(mt,mpred), "F1": f1_score(mt,mpred)}

---
## Grid Search — Hyperparameter Tuning

Search over `C` (regularization strength) and `penalty` (L1 vs L2) using 5-fold CV scored by AUC-ROC.
Best params are used for both Base and Trends models.

In [5]:
param_grid = [
    {"clf__C": [0.01, 0.1, 1.0, 10.0, 100.0], "clf__penalty": ["l2"], "clf__solver": ["lbfgs"]},
    {"clf__C": [0.01, 0.1, 1.0, 10.0, 100.0], "clf__penalty": ["l1"], "clf__solver": ["liblinear"]},
]
_pipe_gs = build_pipeline(NUMERIC_FEATURES, CATEGORICAL_FEATURES)
grid = GridSearchCV(_pipe_gs, param_grid, scoring="roc_auc", cv=5, n_jobs=-1, verbose=1)
grid.fit(train_fit[FEATURES_BASE], y_train, clf__sample_weight=sample_weights)

best_C       = grid.best_params_["clf__C"]
best_solver  = grid.best_params_["clf__solver"]
best_penalty = grid.best_params_["clf__penalty"]
print(f"\nBest params : C={best_C}  penalty={best_penalty}  solver={best_solver}")
print(f"Best CV AUC : {grid.best_score_:.4f}")

Fitting 5 folds for each of 10 candidates, totalling 50 fits

Best params : C=0.01  penalty=l2  solver=lbfgs
Best CV AUC : 0.8642


---
## Baseline — Market Price

The simplest predictor: use `price_at_snapshot` directly as the probability.
This is the crowd's consensus — a useful sanity check for any model we build.

In [6]:
y_prob_baseline        = test_reset["price_at_snapshot"].values
thresh_bl, f1_bl       = get_threshold(y_test, y_prob_baseline)
print(f"Baseline threshold: {thresh_bl:.3f}  |  F1: {f1_bl:.4f}")

metrics_baseline_row = evaluate(y_test, y_prob_baseline, "Baseline (market price)", thresh_bl)
metrics_baseline_mkt = market_eval(test_reset, y_prob_baseline, thresh_bl)
cat_baseline         = per_category(test_reset, y_prob_baseline, thresh_bl)

Baseline threshold: 0.500  |  F1: 0.6699

──────────────────────────────────────────────────
  Baseline (market price)  (threshold=0.500)
──────────────────────────────────────────────────
  AUC-ROC   : 0.8890
  PR-AUC    : 0.7440
  Log-loss  : 0.3278
  Brier     : 0.1004
  Accuracy  : 0.8703
  F1        : 0.6699
──────────────────────────────────────────────────
Market-level evaluation (4,174 markets)
  AUC-ROC  : 0.9420
  PR-AUC   : 0.8283
  Brier    : 0.0652
  Accuracy : 0.9178
  F1       : 0.7351


pipe_base = build_pipeline(
    num_cols=NUMERIC_FEATURES,
    cat_cols=CATEGORICAL_FEATURES,
    C=best_C, solver=best_solver, penalty=best_penalty,
)
pipe_base.fit(train[FEATURES_BASE], y_train, clf__sample_weight=sample_weights)

y_prob_base          = pipe_base.predict_proba(test[FEATURES_BASE])[:, 1]
thresh_base, f1_base = get_threshold(y_test, y_prob_base)
print(f"Optimal threshold: {thresh_base:.3f}  |  F1: {f1_base:.4f}")

## 1.1 Train

In [7]:
pipe_base = build_pipeline(
    num_cols=NUMERIC_FEATURES,
    cat_cols=CATEGORICAL_FEATURES,
    C=best_C, solver=best_solver, penalty=best_penalty,
)
pipe_base.fit(train_fit[FEATURES_BASE], y_train, clf__sample_weight=sample_weights)

# Isotonic calibration on held-out calibration set
p_cal_raw_base = pipe_base.predict_proba(train_cal[FEATURES_BASE])[:, 1]
iso_base = IsotonicRegression(out_of_bounds="clip")
iso_base.fit(p_cal_raw_base, y_cal)

y_prob_base          = iso_base.transform(pipe_base.predict_proba(test[FEATURES_BASE])[:, 1])
thresh_base, f1_base = get_threshold(y_test, y_prob_base)
print(f"Optimal threshold: {thresh_base:.3f}  |  F1: {f1_base:.4f}")

Optimal threshold: 0.426  |  F1: 0.6658


## 1.2 Row-Level Evaluation

In [8]:
metrics_base_row = evaluate(y_test, y_prob_base, "Base Model", thresh_base)


──────────────────────────────────────────────────
  Base Model  (threshold=0.426)
──────────────────────────────────────────────────
  AUC-ROC   : 0.8874
  PR-AUC    : 0.7101
  Log-loss  : 0.3451
  Brier     : 0.0980
  Accuracy  : 0.8576
  F1        : 0.6658
──────────────────────────────────────────────────


## 1.3 Per-Category Breakdown

In [9]:
cat_base = per_category(test_reset, y_prob_base, thresh_base)
cat_base.style.format({"AUC": "{:.4f}", "PR-AUC": "{:.4f}", "F1": "{:.4f}", "YES%": "{:.1%}"})

,n,AUC,PR-AUC,F1,YES%
category,,,,,
geopolitics,16578,0.9355,0.7227,0.6688,13.8%
entertainment,37027,0.9198,0.7139,0.6509,14.9%
finance,25093,0.9171,0.7905,0.7089,24.3%
politics_global,22591,0.9077,0.7260,0.6833,22.3%
politics_us,52699,0.9044,0.7703,0.7303,24.7%
crypto,24586,0.9029,0.7176,0.7188,26.1%
science_tech,15530,0.8849,0.6966,0.6531,17.8%
other,1364,0.8701,0.3549,0.3922,4.7%
sports,93022,0.8427,0.6485,0.5830,20.9%


## 1.4 Market-Level Evaluation

In [10]:
metrics_base_mkt = market_eval(test_reset, y_prob_base, thresh_base)

Market-level evaluation (4,174 markets)
  AUC-ROC  : 0.9386
  PR-AUC   : 0.8085
  Brier    : 0.0648
  Accuracy : 0.9123
  F1       : 0.7332


## 1.5 Feature Importance

In [11]:
clf_step  = pipe_base.named_steps["clf"]
prep_step = pipe_base.named_steps["preprocessor"]
cat_names = list(prep_step.named_transformers_["cat"].get_feature_names_out(CATEGORICAL_FEATURES))
all_names = NUMERIC_FEATURES + cat_names

importance_df_base = (
    pd.DataFrame({"feature": all_names, "coefficient": clf_step.coef_[0]})
    .assign(abs_coef=lambda d: d["coefficient"].abs())
    .sort_values("abs_coef", ascending=False)
    .drop(columns="abs_coef")
    .reset_index(drop=True)
)

importance_df_base.head(20).style.bar(
    subset=["coefficient"], align="zero", color=["#d65f5f", "#5fba7d"]
)

,feature,coefficient
0,price_at_snapshot,1.004976
1,category_other,0.670176
2,duration_days,-0.545861
3,log_volume,0.498261
4,days_before_close,0.464740
5,price_max_14d,0.430758
6,price_min_14d,0.379469
7,category_geopolitics,-0.370502
8,category_sports,-0.290506
9,category_crypto,-0.228116


pipe_trends = build_pipeline(
    num_cols=NUMERIC_FEATURES + TRENDS_FEATURES,
    cat_cols=CATEGORICAL_FEATURES,
    C=best_C, solver=best_solver, penalty=best_penalty,
)
pipe_trends.fit(train[FEATURES_TRENDS], y_train, clf__sample_weight=sample_weights)

y_prob_trends            = pipe_trends.predict_proba(test[FEATURES_TRENDS])[:, 1]
thresh_trends, f1_trends = get_threshold(y_test, y_prob_trends)
print(f"Optimal threshold: {thresh_trends:.3f}  |  F1: {f1_trends:.4f}")

## 2.1 Train

In [12]:
pipe_trends = build_pipeline(
    num_cols=NUMERIC_FEATURES + TRENDS_FEATURES,
    cat_cols=CATEGORICAL_FEATURES,
    C=best_C, solver=best_solver, penalty=best_penalty,
)
pipe_trends.fit(train_fit[FEATURES_TRENDS], y_train, clf__sample_weight=sample_weights)

p_cal_raw_trends = pipe_trends.predict_proba(train_cal[FEATURES_TRENDS])[:, 1]
iso_trends = IsotonicRegression(out_of_bounds="clip")
iso_trends.fit(p_cal_raw_trends, y_cal)

y_prob_trends            = iso_trends.transform(pipe_trends.predict_proba(test[FEATURES_TRENDS])[:, 1])
thresh_trends, f1_trends = get_threshold(y_test, y_prob_trends)
print(f"Optimal threshold: {thresh_trends:.3f}  |  F1: {f1_trends:.4f}")

Optimal threshold: 0.341  |  F1: 0.6655


## 2.2 Row-Level Evaluation

In [13]:
metrics_trends_row = evaluate(y_test, y_prob_trends, "Trends Model", thresh_trends)


──────────────────────────────────────────────────
  Trends Model  (threshold=0.341)
──────────────────────────────────────────────────
  AUC-ROC   : 0.8868
  PR-AUC    : 0.7096
  Log-loss  : 0.3410
  Brier     : 0.0980
  Accuracy  : 0.8531
  F1        : 0.6655
──────────────────────────────────────────────────


## 2.3 Per-Category Breakdown

In [14]:
cat_trends = per_category(test_reset, y_prob_trends, thresh_trends)
cat_trends.style.format({"AUC": "{:.4f}", "PR-AUC": "{:.4f}", "F1": "{:.4f}", "YES%": "{:.1%}"})

,n,AUC,PR-AUC,F1,YES%
category,,,,,
geopolitics,16578,0.9348,0.7219,0.6700,13.8%
entertainment,37027,0.9190,0.7127,0.6472,14.9%
finance,25093,0.9171,0.7925,0.7112,24.3%
politics_global,22591,0.9062,0.7175,0.6876,22.3%
crypto,24586,0.9057,0.7235,0.7205,26.1%
politics_us,52699,0.9035,0.7685,0.7286,24.7%
science_tech,15530,0.8853,0.6933,0.6497,17.8%
other,1364,0.8670,0.3303,0.3759,4.7%
sports,93022,0.8412,0.6484,0.5849,20.9%


## 2.4 Market-Level Evaluation

In [15]:
metrics_trends_mkt = market_eval(test_reset, y_prob_trends, thresh_trends)

Market-level evaluation (4,174 markets)
  AUC-ROC  : 0.9386
  PR-AUC   : 0.8074
  Brier    : 0.0648
  Accuracy : 0.9044
  F1       : 0.7313


## 2.5 Feature Importance

Trend features are highlighted in yellow — their magnitude relative to price features shows how much signal they add.

In [16]:
clf_step  = pipe_trends.named_steps["clf"]
prep_step = pipe_trends.named_steps["preprocessor"]
cat_names = list(prep_step.named_transformers_["cat"].get_feature_names_out(CATEGORICAL_FEATURES))
all_names = NUMERIC_FEATURES + TRENDS_FEATURES + cat_names

importance_df = (
    pd.DataFrame({"feature": all_names, "coefficient": clf_step.coef_[0]})
    .assign(
        abs_coef = lambda d: d["coefficient"].abs(),
        is_trend = lambda d: d["feature"].isin(TRENDS_FEATURES),
    )
    .sort_values("abs_coef", ascending=False)
    .drop(columns="abs_coef")
    .reset_index(drop=True)
)

print("Trend feature coefficients:")
print(importance_df[importance_df["is_trend"]].to_string(index=False))
print()

importance_df.head(25).style.bar(
    subset=["coefficient"], align="zero", color=["#d65f5f", "#5fba7d"]
).apply(
    lambda col: ["background-color: #fff3cd" if v else "" for v in importance_df.head(25)["is_trend"]],
    axis=0, subset=["feature", "coefficient"]
)

Trend feature coefficients:
        feature  coefficient  is_trend
    trend_value     0.266898      True
      trend_ma4    -0.179755      True
trend_change_4w    -0.111699      True
 has_trend_data    -0.053869      True
    trend_spike     0.043883      True



,feature,coefficient,is_trend
0,price_at_snapshot,0.981857,False
1,duration_days,-0.537526,False
2,log_volume,0.503240,False
3,days_before_close,0.457796,False
4,price_max_14d,0.432105,False
5,price_min_14d,0.381720,False
6,trend_value,0.266898,True
7,category_geopolitics,-0.237493,False
8,category_sports,-0.226608,False
9,price_range_14d,0.198644,False


---
# Part 3 — Comparison

Head-to-head: Base vs Trends across row-level metrics, market-level metrics, and per-category AUC.

## 3.1 Row-Level

In [17]:
row_comparison = pd.DataFrame({
    "Baseline": metrics_baseline_row,
    "Base":     metrics_base_row,
    "Trends":   metrics_trends_row,
})
row_comparison["Δ Base"]   = row_comparison["Base"]   - row_comparison["Baseline"]
row_comparison["Δ Trends"] = row_comparison["Trends"] - row_comparison["Baseline"]
row_comparison.style.format("{:.4f}").bar(
    subset=["Δ Base", "Δ Trends"], align="zero", color=["#d65f5f", "#5fba7d"]
)

,Baseline,Base,Trends,Δ Base,Δ Trends
AUC-ROC,0.8890,0.8874,0.8868,-0.0016,-0.0022
PR-AUC,0.7440,0.7101,0.7096,-0.0338,-0.0344
Log-loss,0.3278,0.3451,0.3410,0.0172,0.0132
Brier,0.1004,0.0980,0.0980,-0.0024,-0.0024
Accuracy,0.8703,0.8576,0.8531,-0.0127,-0.0172
F1,0.6699,0.6658,0.6655,-0.0040,-0.0044


## 3.2 Market-Level

In [18]:
mkt_comparison = pd.DataFrame({
    "Baseline": metrics_baseline_mkt,
    "Base":     metrics_base_mkt,
    "Trends":   metrics_trends_mkt,
})
mkt_comparison["Δ Base"]   = mkt_comparison["Base"]   - mkt_comparison["Baseline"]
mkt_comparison["Δ Trends"] = mkt_comparison["Trends"] - mkt_comparison["Baseline"]
mkt_comparison.style.format("{:.4f}").bar(
    subset=["Δ Base", "Δ Trends"], align="zero", color=["#d65f5f", "#5fba7d"]
)

,Baseline,Base,Trends,Δ Base,Δ Trends
AUC-ROC,0.9420,0.9386,0.9386,-0.0034,-0.0035
PR-AUC,0.8283,0.8085,0.8074,-0.0198,-0.0209
Brier,0.0652,0.0648,0.0648,-0.0004,-0.0004
Accuracy,0.9178,0.9123,0.9044,-0.0055,-0.0134
F1,0.7351,0.7332,0.7313,-0.0019,-0.0038


## 3.3 Per-Category AUC Delta

In [19]:
cat_comparison = cat_baseline[["n", "AUC", "YES%"]].rename(columns={"AUC": "AUC (baseline)"})
cat_comparison["AUC (base)"]   = cat_base["AUC"]
cat_comparison["AUC (trends)"] = cat_trends["AUC"]
cat_comparison["Δ Base"]       = cat_comparison["AUC (base)"]   - cat_comparison["AUC (baseline)"]
cat_comparison["Δ Trends"]     = cat_comparison["AUC (trends)"] - cat_comparison["AUC (baseline)"]
(
    cat_comparison
    .sort_values("AUC (baseline)", ascending=False)
    .style
    .format("{:.4f}", subset=["AUC (baseline)", "AUC (base)", "AUC (trends)", "Δ Base", "Δ Trends"])
    .format("{:.1%}", subset=["YES%"])
    .bar(subset=["Δ Base", "Δ Trends"], align="zero", color=["#d65f5f", "#5fba7d"])
)

,n,AUC (baseline),YES%,AUC (base),AUC (trends),Δ Base,Δ Trends
category,,,,,,,
geopolitics,16578,0.9265,13.8%,0.9355,0.9348,0.0090,0.0083
finance,25093,0.9183,24.3%,0.9171,0.9171,-0.0012,-0.0012
entertainment,37027,0.9101,14.9%,0.9198,0.9190,0.0097,0.0089
politics_us,52699,0.9083,24.7%,0.9044,0.9035,-0.0039,-0.0049
politics_global,22591,0.9057,22.3%,0.9077,0.9062,0.0020,0.0005
crypto,24586,0.9052,26.1%,0.9029,0.9057,-0.0024,0.0004
science_tech,15530,0.8876,17.8%,0.8849,0.8853,-0.0027,-0.0023
other,1364,0.8847,4.7%,0.8701,0.8670,-0.0146,-0.0177
sports,93022,0.8400,20.9%,0.8427,0.8412,0.0027,0.0012


---
## Save Artifacts

In [21]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump({"pipeline": pipe_base,   "calibrator": iso_base},   ARTIFACTS_DIR / "model_base.joblib")
joblib.dump({"pipeline": pipe_trends, "calibrator": iso_trends}, ARTIFACTS_DIR / "model_trends.joblib")
print(f"Models saved → {ARTIFACTS_DIR}")

PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
pred_df = test[["market_id", "snapshot_timestamp", "category", TARGET]].copy().reset_index(drop=True)
pred_df["pred_prob_base"]    = y_prob_base
pred_df["pred_prob_trends"]  = y_prob_trends
pred_df["pred_label_base"]   = (y_prob_base   >= thresh_base).astype(int)
pred_df["pred_label_trends"] = (y_prob_trends >= thresh_trends).astype(int)
preds_path = PREDICTIONS_DIR / "predictions.csv"
pred_df.to_csv(preds_path, index=False)
print(f"Predictions saved → {preds_path}")
pred_df.head()

Models saved → artifacts
Predictions saved → predictions/predictions.csv


,market_id,snapshot_timestamp,category,outcome,pred_prob_base,pred_prob_trends,pred_label_base,pred_label_trends
0,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,2023-03-09 21:08:18.864000+00:00,crypto,0,0.465688,0.423354,1,1
1,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,2023-03-10 09:08:18.864000+00:00,crypto,0,0.465688,0.463803,1,1
2,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,2023-03-10 21:08:18.864000+00:00,crypto,0,0.465688,0.463803,1,1
3,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,2023-03-11 09:08:18.864000+00:00,crypto,0,0.465688,0.463803,1,1
4,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,2023-03-11 21:08:18.864000+00:00,crypto,0,0.649843,0.654240,1,1
